In [2]:
"""
ASL Sign Language Recognition - Comprehensive Model Testing
Tests all recommended models from recommended_models.md with proper augmentation

Dataset: GTE9 (205 classes, ~10 videos/class)
Shape: (70, 63, 4) - 70 frames × 63 landmarks × 4 features

Activate venv: source /home/aryan/opensource_lab_proj/venv/bin/activate
"""

import os
os.environ["TF_CPP_MIN_LOG_LEVEL"] = "2"

import numpy as np
import matplotlib.pyplot as plt
import tensorflow as tf
from tensorflow.keras import layers, models
from sklearn.preprocessing import LabelEncoder
from tensorflow.keras.utils import to_categorical
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau, ModelCheckpoint, LearningRateScheduler
from sklearn.model_selection import train_test_split, StratifiedKFold
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay, classification_report
from sklearn.utils.class_weight import compute_class_weight
from datetime import datetime


# ==================== CONFIGURATION ====================
LANDMARK_PATH = "../gte9_landmarks"
MODEL_PATH = "../models"
RESULTS_PATH = "../testing/model_comparison_results"

SEQUENCE_LENGTH = 70
NUM_LANDMARKS = 63
NUM_FEATURES = 4
FEATURE_DIM = NUM_LANDMARKS * NUM_FEATURES  # 252

EPOCHS = 100
BATCH_SIZE = 32  # Increased from 16 for more stable gradients
TEST_SIZE = 0.2
VAL_SIZE = 0.15

# Augmentation settings - INCREASED to combat severe overfitting
AUGMENTATION_FACTOR = 12  # Increased from 8 to 12x for better generalization

# Create results directory
os.makedirs(RESULTS_PATH, exist_ok=True)

# ==================== GPU CONFIGURATION ====================
gpus = tf.config.list_physical_devices("GPU")
if gpus:
    try:
        for gpu in gpus:
            tf.config.experimental.set_memory_growth(gpu, True)
        print(f"✅ GPU detected: {gpus[0].name}")
        
        # Enable mixed precision for faster training
        policy = tf.keras.mixed_precision.Policy('mixed_float16')
        tf.keras.mixed_precision.set_global_policy(policy)
        print(f"✅ Mixed precision enabled: {policy.name}")
    except RuntimeError as e:
        print(f"⚠️ GPU setup error: {e}")
else:
    print("⚠️ No GPU detected, running on CPU.")




✅ GPU detected: /physical_device:GPU:0
✅ Mixed precision enabled: mixed_float16


In [3]:
# ==================== DATA AUGMENTATION ====================
def augment_landmarks(X, augmentation_factor=5, preserve_normalization=True):
    """
    Apply data augmentation while preserving the dataset normalization:
    - Pose landmarks normalized by shoulder width & torso height
    - Hand landmarks normalized similarly
    - Z-coordinates centered at shoulder
    
    Args:
        X: Input data shape (N, 70, 63, 4)
        augmentation_factor: Number of augmented versions per sample
        preserve_normalization: Keep normalization consistent with dataset schema
    
    Returns:
        Augmented data shape (N*augmentation_factor, 70, 63, 4)
    """
    print(f"\n🔄 Applying {augmentation_factor}x data augmentation...")
    print(f"Original shape: {X.shape}")
    
    augmented_data = []
    
    for video in X:
        # Always include original
        augmented_data.append(video)
        
        for _ in range(augmentation_factor - 1):
            aug_video = video.copy()
            
            # 1. ROTATION (around z-axis, ±15 degrees)
            # Rotates in x-y plane while preserving normalization
            angle = np.random.uniform(-15, 15) * np.pi / 180
            cos_a, sin_a = np.cos(angle), np.sin(angle)
            
            x_coords = aug_video[:, :, 0].copy()
            y_coords = aug_video[:, :, 1].copy()
            
            aug_video[:, :, 0] = cos_a * x_coords - sin_a * y_coords
            aug_video[:, :, 1] = sin_a * x_coords + cos_a * y_coords
            
            # 2. SCALING (±10%)
            # Simulates different body sizes while keeping proportions
            scale = np.random.uniform(0.90, 1.10)
            aug_video[:, :, :3] *= scale
            
            # 3. TRANSLATION (small shifts)
            # Simulates different camera positions
            shift_x = np.random.uniform(-0.05, 0.05)
            shift_y = np.random.uniform(-0.05, 0.05)
            aug_video[:, :, 0] += shift_x
            aug_video[:, :, 1] += shift_y
            
            # 4. TEMPORAL JITTERING (time shifts)
            # Simulates different signing speeds
            if np.random.random() > 0.5:
                shift = np.random.randint(-3, 4)
                aug_video = np.roll(aug_video, shift, axis=0)
            
            # 5. GAUSSIAN NOISE (small, realistic)
            # Simulates detection uncertainty
            noise_std = 0.01
            noise = np.random.normal(0, noise_std, aug_video[:, :, :3].shape)
            aug_video[:, :, :3] += noise
            
            # 6. RANDOM FRAME DROPOUT (simulate missing detections)
            # Mirrors real-world detection failures
            if np.random.random() > 0.7:
                num_frames_to_drop = np.random.randint(1, 5)
                frames_to_drop = np.random.choice(70, num_frames_to_drop, replace=False)
                aug_video[frames_to_drop] = 0.0
            
            # 7. TEMPORAL SPEED VARIATION (stretch/compress)
            # Simulates different signing speeds
            # if np.random.random() > 0.6:
            #     speed_factor = np.random.uniform(0.85, 1.15)
            #     new_length = int(70 * speed_factor)
            #     if new_length > 10:  # Ensure reasonable length
            #         indices = np.linspace(0, 69, new_length, dtype=int)
            #         aug_video_temp = aug_video[indices]
            #         # Resample back to 70 frames
            #         final_indices = np.linspace(0, len(aug_video_temp)-1, 70, dtype=int)
            #         aug_video = aug_video_temp[final_indices]
            
            # 8. VISIBILITY PERTURBATION (for pose landmarks)
            # Randomly adjust visibility scores slightly
            if np.random.random() > 0.7:
                vis_noise = np.random.uniform(-0.1, 0.1, aug_video[:, :21, 3].shape)
                aug_video[:, :21, 3] = np.clip(aug_video[:, :21, 3] + vis_noise, 0, 1)
            
            augmented_data.append(aug_video)
    
    result = np.array(augmented_data, dtype=np.float32)
    print(f"✅ Augmented shape: {result.shape}")
    print(f"✅ Augmentation ratio: {augmentation_factor}x")
    
    return result


# ==================== DATA LOADING ====================
def load_data():
    """Load preprocessed GTE9 dataset"""
    print("\n📂 Loading GTE9 dataset...")
    
    if os.path.exists(f"{LANDMARK_PATH}/x.npy"):
        X = np.load(f"{LANDMARK_PATH}/x.npy")
        y = np.load(f"{LANDMARK_PATH}/y.npy")
        y_encoded = np.load(f"{LANDMARK_PATH}/y_encoded.npy")
        y_onehot = np.load(f"{LANDMARK_PATH}/y_onehot.npy")
        
        label_encoder = LabelEncoder()
        label_encoder.fit(y)
        
        print(f"✅ Loaded from cache:")
        print(f"   X shape: {X.shape}")
        print(f"   y shape: {y.shape}")
        print(f"   Classes: {len(label_encoder.classes_)}")
        print(f"   Samples per class: ~{len(X) / len(label_encoder.classes_):.1f}")
        
        return X, y, y_encoded, y_onehot, label_encoder
    else:
        raise FileNotFoundError(f"Preprocessed data not found in {LANDMARK_PATH}")



In [4]:

# ==================== MODEL ARCHITECTURES ====================

def build_lightweight_bilstm(num_classes, name="Lightweight_BiLSTM"):
    """
    Lightweight Bi-LSTM - Optimized version
    Moderate regularization with better capacity
    Expected accuracy: 45-60% with augmentation
    """
    model = models.Sequential([
        layers.Input(shape=(SEQUENCE_LENGTH, FEATURE_DIM)),
        
        layers.Bidirectional(layers.LSTM(128, return_sequences=True)),
        layers.BatchNormalization(),
        layers.Dropout(0.35),
        
        layers.Bidirectional(layers.LSTM(64)),
        layers.BatchNormalization(),
        layers.Dropout(0.35),
        
        layers.Dense(128, activation='relu', 
                    kernel_regularizer=tf.keras.regularizers.l2(0.003)),
        layers.BatchNormalization(),
        layers.Dropout(0.3),
        
        layers.Dense(num_classes, activation='softmax', dtype='float32')
    ], name=name)
    
    return model


def build_lightweight_bilstm_balanced_regularization(num_classes, name="Lightweight_BiLSTM_Balanced_Regularization"):
    """
    ⭐ RECOMMENDED FOR SMALL DATASETS ⭐
    Lightweight Bi-LSTM with VERY STRONG regularization (FAST VERSION - no recurrent_dropout)
    Designed to combat SEVERE overfitting on tiny datasets (204 classes, ~10 samples/class)
    Expected accuracy: 35-50% with augmentation (improved from 26%)
    Training speed: ~40-60ms/step (5-8x faster than version with recurrent_dropout)
    """
    model = models.Sequential([
        layers.Input(shape=(SEQUENCE_LENGTH, FEATURE_DIM)),
        
        # Reduced capacity: 128→64, 64→32
        layers.Bidirectional(layers.LSTM(64, return_sequences=True,
                                         kernel_regularizer=tf.keras.regularizers.l2(0.01))),
        layers.BatchNormalization(),
        layers.Dropout(0.6),  # Increased from 0.5
        
        layers.Bidirectional(layers.LSTM(32,
                                         kernel_regularizer=tf.keras.regularizers.l2(0.01))),
        layers.BatchNormalization(),
        layers.Dropout(0.6),  # Increased from 0.5
        
        # Reduced dense layer: 128→64
        layers.Dense(64, activation='relu', 
                    kernel_regularizer=tf.keras.regularizers.l2(0.015)),
        layers.BatchNormalization(),
        layers.Dropout(0.6),  # Increased from 0.5
        
        layers.Dense(num_classes, activation='softmax', dtype='float32')
    ], name=name)
    
    return model


def build_bigru_model(num_classes, name="BiGRU"):
    """
    Bi-GRU model - faster training than LSTM
    Optimized with stronger regularization to prevent overfitting
    Expected accuracy: 50-65% with augmentation
    """
    model = models.Sequential([
        layers.Input(shape=(SEQUENCE_LENGTH, FEATURE_DIM)),
        
        layers.Bidirectional(layers.GRU(128, return_sequences=True,
                                        kernel_regularizer=tf.keras.regularizers.l2(0.002))),
        layers.BatchNormalization(),
        layers.Dropout(0.45),
        
        layers.Bidirectional(layers.GRU(64,
                                        kernel_regularizer=tf.keras.regularizers.l2(0.002))),
        layers.BatchNormalization(),
        layers.Dropout(0.45),
        
        layers.Dense(128, activation='relu',
                    kernel_regularizer=tf.keras.regularizers.l2(0.003)),
        layers.BatchNormalization(),
        layers.Dropout(0.4),
        
        layers.Dense(num_classes, activation='softmax', dtype='float32')
    ], name=name)
    
    return model


def build_bigru_balanced_regularization(num_classes, name="BiGRU_Balanced_Regularization"):
    """
    ⭐ RECOMMENDED FOR SMALL DATASETS ⭐
    Bi-GRU model with STRONG regularization (FAST VERSION - no recurrent_dropout)
    Tuned to reduce over-regularization: slightly lower dropout and L2.
    Expected accuracy: 38-45% with augmentation
    Training speed: ~35-50ms/step
    """
    model = models.Sequential([
        layers.Input(shape=(SEQUENCE_LENGTH, FEATURE_DIM)),
        
        # Reduced capacity: 128→64, 64→32, with slightly reduced L2
        layers.Bidirectional(layers.GRU(64, return_sequences=True,
                                        kernel_regularizer=tf.keras.regularizers.l2(0.008))),
        layers.BatchNormalization(),
        layers.Dropout(0.55),  # was 0.6
        
        layers.Bidirectional(layers.GRU(32,
                                        kernel_regularizer=tf.keras.regularizers.l2(0.008))),
        layers.BatchNormalization(),
        layers.Dropout(0.55),  # was 0.6
        
        # Reduced dense layer L2
        layers.Dense(64, activation='relu',
                    kernel_regularizer=tf.keras.regularizers.l2(0.010)),  # was 0.015
        layers.BatchNormalization(),
        layers.Dropout(0.55),  # was 0.6
        
        layers.Dense(num_classes, activation='softmax', dtype='float32')
    ], name=name)
    
    return model


def build_lstm_attention(num_classes, name="LSTM_Attention"):
    """
    Bi-LSTM with attention mechanism
    Optimized with VERY STRONG regularization for tiny datasets
    Expected accuracy: 40-55% with augmentation (improved from 37%)
    """
    inputs = layers.Input(shape=(SEQUENCE_LENGTH, FEATURE_DIM))
    
    # Bidirectional LSTM layers with STRONGER regularization
    # Reduced capacity: 128→64, 64→32
    x = layers.Bidirectional(layers.LSTM(64, return_sequences=True,
                                         kernel_regularizer=tf.keras.regularizers.l2(0.01)))(inputs)
    x = layers.BatchNormalization()(x)
    x = layers.Dropout(0.6)(x)  # Increased from 0.35
    
    x = layers.Bidirectional(layers.LSTM(32, return_sequences=True,
                                         kernel_regularizer=tf.keras.regularizers.l2(0.01)))(x)
    x = layers.BatchNormalization()(x)
    x = layers.Dropout(0.6)(x)  # Increased from 0.35
    
    # Attention mechanism
    attention = layers.Dense(1, activation='tanh')(x)
    attention = layers.Flatten()(attention)
    attention = layers.Activation('softmax')(attention)
    attention = layers.RepeatVector(64)(attention)  # Changed from 128
    attention = layers.Permute([2, 1])(attention)
    
    # Apply attention
    x = layers.Multiply()([x, attention])
    x = layers.Lambda(lambda x: tf.reduce_sum(x, axis=1))(x)
    
    # Classification head - reduced capacity
    x = layers.Dense(64, activation='relu',
                    kernel_regularizer=tf.keras.regularizers.l2(0.015))(x)
    x = layers.Dropout(0.6)(x)  # Increased from 0.4
    outputs = layers.Dense(num_classes, activation='softmax', dtype='float32')(x)
    
    model = models.Model(inputs=inputs, outputs=outputs, name=name)
    return model


def build_3dcnn_model(num_classes, name="3D_CNN"):
    """
    3D CNN for spatial-temporal features
    Expected accuracy: 35-50% with augmentation
    ⚠️ High overfitting risk - NOT recommended for small datasets
    """
    inputs = layers.Input(shape=(SEQUENCE_LENGTH, NUM_LANDMARKS, NUM_FEATURES, 1))
    
    # First 3D Conv block
    x = layers.Conv3D(32, kernel_size=(3, 3, 2), activation='relu', padding='same')(inputs)
    x = layers.BatchNormalization()(x)
    x = layers.MaxPooling3D(pool_size=(2, 2, 1))(x)
    x = layers.Dropout(0.4)(x)
    
    # Second 3D Conv block
    x = layers.Conv3D(64, kernel_size=(3, 3, 2), activation='relu', padding='same')(x)
    x = layers.BatchNormalization()(x)
    x = layers.MaxPooling3D(pool_size=(2, 2, 1))(x)
    x = layers.Dropout(0.4)(x)
    
    # Flatten and dense layers
    x = layers.Flatten()(x)
    x = layers.Dense(256, activation='relu')(x)
    x = layers.BatchNormalization()(x)
    x = layers.Dropout(0.5)(x)
    
    x = layers.Dense(128, activation='relu')(x)
    x = layers.Dropout(0.5)(x)
    
    outputs = layers.Dense(num_classes, activation='softmax', dtype='float32')(x)
    
    model = models.Model(inputs=inputs, outputs=outputs, name=name)
    return model


def build_small_transformer(num_classes, d_model=64, num_heads=4, num_layers=1, name="Small_Transformer"):
    """
    Ultra-lightweight Transformer for tiny datasets
    Reduced from d_model=128 to 64, num_layers=2 to 1
    Expected accuracy: 35-50% with augmentation (improved from 30%)
    ⚠️ High overfitting risk even with heavy regularization
    """
    inputs = layers.Input(shape=(SEQUENCE_LENGTH, FEATURE_DIM))
    
    # Positional encoding
    positions = tf.range(start=0, limit=SEQUENCE_LENGTH, delta=1)
    position_embedding = layers.Embedding(input_dim=SEQUENCE_LENGTH, output_dim=FEATURE_DIM)(positions)
    x = inputs + position_embedding
    
    # Project to d_model dimensions
    x = layers.Dense(d_model, kernel_regularizer=tf.keras.regularizers.l2(0.01))(x)
    
    # Reduced to 1 transformer layer (was 2)
    for _ in range(num_layers):
        # Multi-head attention with higher dropout
        attn_output = layers.MultiHeadAttention(
            num_heads=num_heads,
            key_dim=d_model // num_heads,
            dropout=0.4  # Increased from 0.2
        )(x, x)
        x = layers.Add()([x, attn_output])
        x = layers.LayerNormalization(epsilon=1e-6)(x)
        
        # Feed-forward network with stronger regularization
        ffn = models.Sequential([
            layers.Dense(d_model * 2, activation='relu',
                        kernel_regularizer=tf.keras.regularizers.l2(0.01)),
            layers.Dropout(0.5),  # Increased from 0.2
            layers.Dense(d_model, kernel_regularizer=tf.keras.regularizers.l2(0.01))
        ])
        ffn_output = ffn(x)
        x = layers.Add()([x, ffn_output])
        x = layers.LayerNormalization(epsilon=1e-6)(x)
    
    # Global average pooling
    x = layers.GlobalAveragePooling1D()(x)
    
    # Simplified classification head (removed one dense layer)
    x = layers.Dense(64, activation='relu',
                    kernel_regularizer=tf.keras.regularizers.l2(0.015))(x)
    x = layers.Dropout(0.6)(x)  # Increased from 0.4
    
    outputs = layers.Dense(num_classes, activation='softmax', dtype='float32')(x)
    
    model = models.Model(inputs=inputs, outputs=outputs, name=name)
    return model


def build_hybrid_cnn_lstm(num_classes, name="Hybrid_CNN_LSTM"):
    """
    Hybrid CNN-LSTM combining spatial and temporal features
    Expected accuracy: 50-65% with augmentation
    ⚠️ High complexity - may overfit on small datasets
    """
    inputs = layers.Input(shape=(SEQUENCE_LENGTH, NUM_LANDMARKS, NUM_FEATURES))
    
    # Treat each frame as 2D spatial data
    # TimeDistributed CNN for spatial feature extraction
    x = layers.Reshape((SEQUENCE_LENGTH, NUM_LANDMARKS, NUM_FEATURES, 1))(inputs)
    
    # Apply 2D convolutions per frame
    x = layers.TimeDistributed(
        layers.Conv2D(32, (3, 3), activation='relu', padding='same')
    )(x)
    x = layers.TimeDistributed(layers.MaxPooling2D((2, 1)))(x)
    x = layers.TimeDistributed(layers.Dropout(0.3))(x)
    
    x = layers.TimeDistributed(
        layers.Conv2D(64, (3, 3), activation='relu', padding='same')
    )(x)
    x = layers.TimeDistributed(layers.GlobalAveragePooling2D())(x)
    
    # LSTM for temporal modeling
    x = layers.Bidirectional(layers.LSTM(128, return_sequences=True))(x)
    x = layers.Dropout(0.4)(x)
    
    x = layers.Bidirectional(layers.LSTM(64))(x)
    x = layers.Dropout(0.4)(x)
    
    # Classification
    x = layers.Dense(128, activation='relu')(x)
    x = layers.Dropout(0.4)(x)
    outputs = layers.Dense(num_classes, activation='softmax', dtype='float32')(x)
    
    model = models.Model(inputs=inputs, outputs=outputs, name=name)
    return model



In [5]:

# ==================== TRAINING UTILITIES ====================

def get_callbacks(model_name):
    """Create training callbacks"""
    early_stop = EarlyStopping(
        monitor='val_loss',  # Monitor loss for better overfitting detection
        patience=15,  # Reduced patience to stop overfitting earlier
        mode='min',
        restore_best_weights=True,
        verbose=1
    )
    
    reduce_lr = ReduceLROnPlateau(
        monitor='val_loss',
        factor=0.5,
        patience=5,  # Reduce LR faster when overfitting
        mode='min',
        min_lr=1e-7,
        verbose=1
    )
    
    checkpoint = ModelCheckpoint(
        f"{RESULTS_PATH}/{model_name}_best.keras",
        monitor='val_accuracy',
        save_best_only=True,
        mode='max',
        verbose=1
    )
    
    return [early_stop, reduce_lr, checkpoint]


def train_model(model, X_train, y_train, X_val, y_val, model_name, class_weight_dict):
    """Train a model and return history"""
    print(f"\n{'='*60}")
    print(f"Training: {model_name}")
    print(f"{'='*60}")
    
    model.compile(
        optimizer=tf.keras.optimizers.AdamW(learning_rate=0.00035, weight_decay=0.008),  # tuned LR + weight decay
        loss=tf.keras.losses.CategoricalCrossentropy(label_smoothing=0.1),  # keep smoothing
        metrics=['accuracy']
    )
    
    model.summary()
    
    history = model.fit(
        X_train, y_train,
        validation_data=(X_val, y_val),
        batch_size=BATCH_SIZE,
        epochs=EPOCHS,
        callbacks=get_callbacks(model_name),
        class_weight=class_weight_dict,
        verbose=1
    )
    
    return history


def evaluate_model(model, X_test, y_test, label_encoder, model_name):
    """Evaluate model and generate reports"""
    print(f"\n{'='*60}")
    print(f"Evaluating: {model_name}")
    print(f"{'='*60}")
    
    # Predictions
    y_pred = model.predict(X_test, verbose=0)
    y_pred_labels = np.argmax(y_pred, axis=1)
    y_true_labels = np.argmax(y_test, axis=1)
    
    # Calculate accuracy
    loss, accuracy = model.evaluate(X_test, y_test, verbose=0)
    print(f"✅ Test Accuracy: {accuracy*100:.2f}%")
    print(f"✅ Test Loss: {loss:.4f}")
    
    # Classification report
    print("\nClassification Report:")
    print(classification_report(
        y_true_labels, y_pred_labels,
        target_names=label_encoder.classes_,
        zero_division=0,
        digits=3
    ))
    
    # Confusion matrix
    cm = confusion_matrix(y_true_labels, y_pred_labels)
    
    return accuracy, loss, cm, y_pred_labels, y_true_labels


def plot_results(history, cm, label_encoder, model_name):
    """Generate visualization plots"""
    timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
    
    # Training curves
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    
    axes[0].plot(history.history['accuracy'], label='Train Acc', linewidth=2)
    axes[0].plot(history.history['val_accuracy'], label='Val Acc', linewidth=2)
    axes[0].set_xlabel('Epoch', fontsize=12)
    axes[0].set_ylabel('Accuracy', fontsize=12)
    axes[0].set_title(f'{model_name} - Accuracy', fontsize=14, fontweight='bold')
    axes[0].legend()
    axes[0].grid(True, alpha=0.3)
    
    axes[1].plot(history.history['loss'], label='Train Loss', linewidth=2)
    axes[1].plot(history.history['val_loss'], label='Val Loss', linewidth=2)
    axes[1].set_xlabel('Epoch', fontsize=12)
    axes[1].set_ylabel('Loss', fontsize=12)
    axes[1].set_title(f'{model_name} - Loss', fontsize=14, fontweight='bold')
    axes[1].legend()
    axes[1].grid(True, alpha=0.3)
    
    plt.tight_layout()
    plt.savefig(f"{RESULTS_PATH}/{model_name}_{timestamp}_training_curves.png", dpi=150)
    plt.close()
    
    # Confusion matrix
    plt.figure(figsize=(20, 18))
    disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=label_encoder.classes_)
    disp.plot(xticks_rotation=90, cmap='Blues', values_format='d')
    plt.title(f'{model_name} - Confusion Matrix', fontsize=16, fontweight='bold', pad=20)
    plt.tight_layout()
    plt.savefig(f"{RESULTS_PATH}/{model_name}_{timestamp}_confusion_matrix.png", dpi=150)
    plt.close()
    
    print(f"✅ Plots saved to {RESULTS_PATH}/")



In [6]:

# ==================== MAIN EXECUTION ====================


"""Main execution function"""
print("\n" + "="*80)
print("ASL SIGN LANGUAGE RECOGNITION - COMPREHENSIVE MODEL TESTING")
print("="*80)

# Load data
X, y, y_encoded, y_onehot, label_encoder = load_data()
num_classes = len(label_encoder.classes_)

print(f"\n📊 Dataset Statistics:")
print(f"   Total samples: {len(X)}")
print(f"   Number of classes: {num_classes}")
print(f"   Samples per class: {len(X) / num_classes:.1f}")
print(f"   Data shape (loaded): {X.shape}")

# Reshape from (N, 70, 252) to (N, 70, 63, 4) for augmentation
X = X.reshape(-1, SEQUENCE_LENGTH, NUM_LANDMARKS, NUM_FEATURES)
print(f"   Data shape (reshaped): {X.shape}")

# Split data
X_train, X_test, y_train, y_test = train_test_split(
    X, y_onehot, test_size=TEST_SIZE, random_state=123, stratify=y_encoded
)

X_train, X_val, y_train, y_val = train_test_split(
    X_train, y_train, test_size=VAL_SIZE, stratify=np.argmax(y_train, axis=1), random_state=123
)

print(f"\n📊 Data Split:")
print(f"   Training: {X_train.shape[0]} samples, shape: {X_train.shape}")
print(f"   Validation: {X_val.shape[0]} samples, shape: {X_val.shape}")
print(f"   Test: {X_test.shape[0]} samples, shape: {X_test.shape}")

# Apply augmentation ONLY to training data
# X_train/val/test are already in shape (N, 70, 63, 4)
X_train_aug = augment_landmarks(X_train, augmentation_factor=AUGMENTATION_FACTOR)
y_train_aug = np.repeat(y_train, AUGMENTATION_FACTOR, axis=0)

print(f"\n✅ Training data after augmentation: {X_train_aug.shape[0]} samples")
print(f"   Shape: {X_train_aug.shape}")

# Reshape for LSTM/GRU models (flatten landmarks)
X_train_flat = X_train_aug.reshape(-1, SEQUENCE_LENGTH, FEATURE_DIM)
X_val_flat = X_val.reshape(-1, SEQUENCE_LENGTH, FEATURE_DIM)
X_test_flat = X_test.reshape(-1, SEQUENCE_LENGTH, FEATURE_DIM)

print(f"\n📊 Flattened data for LSTM/GRU/Transformer:")
print(f"   Training: {X_train_flat.shape}")
print(f"   Validation: {X_val_flat.shape}")
print(f"   Test: {X_test_flat.shape}")

# Keep 4D shape for 3D CNN and hybrid models
X_train_3d = X_train_aug  # (N, 70, 63, 4)
X_val_3d = X_val  # (N, 70, 63, 4)
X_test_3d = X_test  # (N, 70, 63, 4)

print(f"\n📊 4D data for 3D CNN and Hybrid models:")
print(f"   Training: {X_train_3d.shape}")
print(f"   Validation: {X_val_3d.shape}")
print(f"   Test: {X_test_3d.shape}")

# Compute class weights
y_train_labels = np.argmax(y_train_aug, axis=1)
class_weights = compute_class_weight(
    'balanced',
    classes=np.unique(y_train_labels),
    y=y_train_labels
)
class_weight_dict = dict(enumerate(class_weights))
print(f"\n✅ Class weights computed (min={min(class_weights):.3f}, max={max(class_weights):.3f})")

# Store results
results = {}



ASL SIGN LANGUAGE RECOGNITION - COMPREHENSIVE MODEL TESTING

📂 Loading GTE9 dataset...
✅ Loaded from cache:
   X shape: (2060, 70, 252)
   y shape: (2060,)
   Classes: 204
   Samples per class: ~10.1

📊 Dataset Statistics:
   Total samples: 2060
   Number of classes: 204
   Samples per class: 10.1
   Data shape (loaded): (2060, 70, 252)
   Data shape (reshaped): (2060, 70, 63, 4)

📊 Data Split:
   Training: 1400 samples, shape: (1400, 70, 63, 4)
   Validation: 248 samples, shape: (248, 70, 63, 4)
   Test: 412 samples, shape: (412, 70, 63, 4)

🔄 Applying 12x data augmentation...
Original shape: (1400, 70, 63, 4)
✅ Augmented shape: (16800, 70, 63, 4)
✅ Augmentation ratio: 12x

✅ Training data after augmentation: 16800 samples
   Shape: (16800, 70, 63, 4)

📊 Flattened data for LSTM/GRU/Transformer:
   Training: (16800, 70, 252)
   Validation: (248, 70, 252)
   Test: (412, 70, 252)

📊 4D data for 3D CNN and Hybrid models:
   Training: (16800, 70, 63, 4)
   Validation: (248, 70, 63, 4)
   

In [7]:

# ==================== TEST MODELS ====================

# # 1. Lightweight Bi-LSTM (Aggressive Regularization)
# print("\n" + "="*80)
# print("1️⃣  Testing: Lightweight Bi-LSTM - Aggressive Regularization")
# print("="*80)
# model = build_lightweight_bilstm(num_classes)
# history = train_model(model, X_train_flat, y_train_aug, X_val_flat, y_val, 
#                      "Lightweight_BiLSTM", class_weight_dict)
# acc, loss, cm, y_pred, y_true = evaluate_model(model, X_test_flat, y_test, 
#                                                 label_encoder, "Lightweight_BiLSTM")
# plot_results(history, cm, label_encoder, "Lightweight_BiLSTM")
# results['Lightweight_BiLSTM'] = {'accuracy': acc, 'loss': loss}
# del model


In [ ]:

# 2. Lightweight Bi-LSTM (Balanced Regularization) ⭐ RECOMMENDED
print("\n" + "="*80)
print("2️⃣  Testing: Lightweight Bi-LSTM - Balanced Regularization ⭐ RECOMMENDED")
print("="*80)
model = build_lightweight_bilstm_balanced_regularization(num_classes)
history = train_model(model, X_train_flat, y_train_aug, X_val_flat, y_val, 
                        "Lightweight_BiLSTM_Balanced_Regularization", class_weight_dict)
acc, loss, cm, y_pred, y_true = evaluate_model(model, X_test_flat, y_test, 
                                                label_encoder, "Lightweight_BiLSTM_Balanced_Regularization")
plot_results(history, cm, label_encoder, "Lightweight_BiLSTM_Balanced_Regularization")
results['Lightweight_BiLSTM_Balanced_Regularization'] = {'accuracy': acc, 'loss': loss}
del model



2️⃣  Testing: Lightweight Bi-LSTM - Balanced Regularization ⭐ RECOMMENDED


I0000 00:00:1764195730.558895  133427 gpu_device.cc:2020] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 3537 MB memory:  -> device: 0, name: NVIDIA GeForce RTX 4050 Laptop GPU, pci bus id: 0000:01:00.0, compute capability: 8.9



Training: Lightweight_BiLSTM_Balanced_Regularization


Model: "Lightweight_BiLSTM_Balanced_Regularization"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ bidirectional (Bidirectional)   │ (None, 70, 128)        │       162,304 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization             │ (None, 70, 128)        │           512 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 70, 128)        │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ bidirectional_1 (Bidirectional) │ (None, 64)             │        41,216 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_1           │ (None, 64)             │           256 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_1 (Dropout)             │ (None, 64)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 64)             │         4,160 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_2           │ (None, 64)             │           256 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_2 (Dropout)             │ (None, 64)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 204)            │        13,260 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

...

Epoch 1/100


2025-11-27 03:52:22.248897: E tensorflow/core/util/util.cc:131] oneDNN supports DT_HALF only on platforms with AVX-512. Falling back to the default Eigen-based implementation if present.


524/525 ━━━━━━━━━━━━━━━━━━━━ 0s 45ms/step - accuracy: 0.0075 - loss: 12.5926
Epoch 1: val_accuracy improved from None to 0.00806, saving model to ../testing/model_comparison_results/Lightweight_BiLSTM_Balanced_Regularization_best.keras
525/525 ━━━━━━━━━━━━━━━━━━━━ 37s 53ms/step - accuracy: 0.0073 - loss: 11.2323 - val_accuracy: 0.0081 - val_loss: 8.9280 - learning_rate: 3.5000e-04
Epoch 2/100
524/525 ━━━━━━━━━━━━━━━━━━━━ 0s 40ms/step - accuracy: 0.0065 - loss: 8.6425
Epoch 2: val_accuracy improved from 0.00806 to 0.01210, saving model to ../testing/model_comparison_results/Lightweight_BiLSTM_Balanced_Regularization_best.keras
525/525 ━━━━━━━━━━━━━━━━━━━━ 21s 40ms/step - accuracy: 0.0076 - loss: 8.1449 - val_accuracy: 0.0121 - val_loss: 7.1732 - learning_rate: 3.5000e-04
Epoch 3/100
525/525 ━━━━━━━━━━━━━━━━━━━━ 0s 39ms/step - accuracy: 0.0127 - loss: 6.9951
Epoch 3: val_accuracy did not improve from 0.01210
525/525 ━━━━━━━━━━━━━━━━━━━━ 21s 40ms/step - accuracy: 0.0126 - loss: 6.7396 - v

KeyboardInterrupt: 

: 

In [ ]:

# # 3. Bi-GRU (Original)
# print("\n" + "="*80)
# print("3️⃣  Testing: Bi-GRU - Original")
# print("="*80)
# model = build_bigru_model(num_classes)
# history = train_model(model, X_train_flat, y_train_aug, X_val_flat, y_val,
#                      "BiGRU", class_weight_dict)
# acc, loss, cm, y_pred, y_true = evaluate_model(model, X_test_flat, y_test,
#                                                 label_encoder, "BiGRU")
# plot_results(history, cm, label_encoder, "BiGRU")
# results['BiGRU'] = {'accuracy': acc, 'loss': loss}
# del model


In [ ]:

# 4. Bi-GRU (Balanced Regularization) ⭐ RECOMMENDED
print("\n" + "="*80)
print("4️⃣  Testing: Bi-GRU - Balanced Regularization ⭐ RECOMMENDED")
print("="*80)
model = build_bigru_balanced_regularization(num_classes)
history = train_model(model, X_train_flat, y_train_aug, X_val_flat, y_val,
                        "BiGRU_Balanced_Regularization", class_weight_dict)
acc, loss, cm, y_pred, y_true = evaluate_model(model, X_test_flat, y_test,
                                                label_encoder, "BiGRU_Balanced_Regularization")
plot_results(history, cm, label_encoder, "BiGRU_Balanced_Regularization")
results['BiGRU_Balanced_Regularization'] = {'accuracy': acc, 'loss': loss}
del model



4️⃣  Testing: Bi-GRU - Balanced Regularization ⭐ RECOMMENDED

Training: BiGRU_Balanced_Regularization

Training: BiGRU_Balanced_Regularization


Model: "BiGRU_Balanced_Regularization"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ bidirectional_2 (Bidirectional) │ (None, 70, 128)        │       122,112 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_3           │ (None, 70, 128)        │           512 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_3 (Dropout)             │ (None, 70, 128)        │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ bidirectional_3 (Bidirectional) │ (None, 64)             │        31,104 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_4           │ (None, 64)             │           256 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_4 (Dropout)             │ (None, 64)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_2 (Dense)                 │ (None, 64)             │         4,160 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_5           │ (None, 64)             │           256 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_5 (Dropout)             │ (None, 64)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_3 (Dense)                 │ (None, 204)            │        13,260 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 171,660 (670.55 KB)

 Trainable params: 171,148 (668.55 KB)

 Non-trainable params: 512 (2.00 KB)

Epoch 1/100
525/525 ━━━━━━━━━━━━━━━━━━━━ 0s 38ms/step - accuracy: 0.0058 - loss: 11.4533
Epoch 1: val_accuracy improved from None to 0.00806, saving model to ../testing/model_comparison_results/BiGRU_Balanced_Regularization_best.keras

Epoch 1: val_accuracy improved from None to 0.00806, saving model to ../testing/model_comparison_results/BiGRU_Balanced_Regularization_best.keras
525/525 ━━━━━━━━━━━━━━━━━━━━ 24s 40ms/step - accuracy: 0.0060 - loss: 10.2025 - val_accuracy: 0.0081 - val_loss: 8.2574 - learning_rate: 3.0000e-04
Epoch 2/100
525/525 ━━━━━━━━━━━━━━━━━━━━ 24s 40ms/step - accuracy: 0.0060 - loss: 10.2025 - val_accuracy: 0.0081 - val_loss: 8.2574 - learning_rate: 3.0000e-04
Epoch 2/100
524/525 ━━━━━━━━━━━━━━━━━━━━ 0s 40ms/step - accuracy: 0.0064 - loss: 8.1432
Epoch 2: val_accuracy did not improve from 0.00806
525/525 ━━━━━━━━━━━━━━━━━━━━ 21s 40ms/step - accuracy: 0.0067 - loss: 7.8545 - val_accuracy: 0.0040 - val_loss: 7.1616 - learning_rate: 3.0000e-04
Epoch 3/100

Epoch 2: va

<Figure size 2000x1800 with 0 Axes>

In [ ]:

# # 5. LSTM with Attention
# print("\n" + "="*80)
# print("5️⃣  Testing: LSTM with Attention")
# print("="*80)
# model = build_lstm_attention(num_classes)
# history = train_model(model, X_train_flat, y_train_aug, X_val_flat, y_val,
#                         "LSTM_Attention", class_weight_dict)
# acc, loss, cm, y_pred, y_true = evaluate_model(model, X_test_flat, y_test,
#                                                 label_encoder, "LSTM_Attention")
# plot_results(history, cm, label_encoder, "LSTM_Attention")
# results['LSTM_Attention'] = {'accuracy': acc, 'loss': loss}
# del model


In [ ]:

# # 6. Small Transformer
# print("\n" + "="*80)
# print("6️⃣  Testing: Small Transformer")
# print("="*80)
# model = build_small_transformer(num_classes)
# history = train_model(model, X_train_flat, y_train_aug, X_val_flat, y_val,
#                         "Small_Transformer", class_weight_dict)
# acc, loss, cm, y_pred, y_true = evaluate_model(model, X_test_flat, y_test,
#                                                 label_encoder, "Small_Transformer")
# plot_results(history, cm, label_encoder, "Small_Transformer")
# results['Small_Transformer'] = {'accuracy': acc, 'loss': loss}
# del model


In [ ]:

# # 5. 3D CNN (reshape needed)
# print("\n" + "="*80)
# print("5️⃣  Testing: 3D CNN (⚠️ High overfitting risk)")
# print("="*80)
# X_train_cnn = X_train_3d[..., np.newaxis]
# X_val_cnn = X_val_3d[..., np.newaxis]
# X_test_cnn = X_test_3d[..., np.newaxis]

# model = build_3dcnn_model(num_classes)
# history = train_model(model, X_train_cnn, y_train_aug, X_val_cnn, y_val,
#                      "3D_CNN", class_weight_dict)
# acc, loss, cm, y_pred, y_true = evaluate_model(model, X_test_cnn, y_test,
#                                                 label_encoder, "3D_CNN")
# plot_results(history, cm, label_encoder, "3D_CNN")
# results['3D_CNN'] = {'accuracy': acc, 'loss': loss}
# del model

# # 6. Hybrid CNN-LSTM
# print("\n" + "="*80)
# print("6️⃣  Testing: Hybrid CNN-LSTM")
# print("="*80)
# model = build_hybrid_cnn_lstm(num_classes)
# history = train_model(model, X_train_3d, y_train_aug, X_val_3d, y_val,
#                      "Hybrid_CNN_LSTM", class_weight_dict)
# acc, loss, cm, y_pred, y_true = evaluate_model(model, X_test_3d, y_test,
#                                                 label_encoder, "Hybrid_CNN_LSTM")
# plot_results(history, cm, label_encoder, "Hybrid_CNN_LSTM")
# results['Hybrid_CNN_LSTM'] = {'accuracy': acc, 'loss': loss}
# del model


In [ ]:

# ==================== SUMMARY ====================
print("\n" + "="*80)
print("📊 FINAL RESULTS SUMMARY")
print("="*80)

# Sort by accuracy
sorted_results = sorted(results.items(), key=lambda x: x[1]['accuracy'], reverse=True)

print(f"\n{'Rank':<6} {'Model':<25} {'Accuracy':<12} {'Loss':<10}")
print("-" * 60)
for rank, (model_name, metrics) in enumerate(sorted_results, 1):
    print(f"{rank:<6} {model_name:<25} {metrics['accuracy']*100:>6.2f}%     {metrics['loss']:>6.4f}")

# Save summary
summary_file = f"{RESULTS_PATH}/summary_{datetime.now().strftime('%Y%m%d_%H%M%S')}.txt"
with open(summary_file, 'w') as f:
    f.write("ASL SIGN LANGUAGE RECOGNITION - MODEL COMPARISON RESULTS\n")
    f.write("="*80 + "\n\n")
    f.write(f"Dataset: GTE9 ({num_classes} classes)\n")
    f.write(f"Augmentation: {AUGMENTATION_FACTOR}x\n")
    f.write(f"Training samples: {X_train_aug.shape[0]}\n")
    f.write(f"Test samples: {X_test.shape[0]}\n\n")
    f.write(f"{'Rank':<6} {'Model':<25} {'Accuracy':<12} {'Loss':<10}\n")
    f.write("-" * 60 + "\n")
    for rank, (model_name, metrics) in enumerate(sorted_results, 1):
        f.write(f"{rank:<6} {model_name:<25} {metrics['accuracy']*100:>6.2f}%     {metrics['loss']:>6.4f}\n")

print(f"\n✅ Summary saved to: {summary_file}")
print(f"✅ All results saved to: {RESULTS_PATH}/")
print("\n" + "="*80)
print("🎉 Testing Complete!")
print("="*80)





📊 FINAL RESULTS SUMMARY

Rank   Model                     Accuracy     Loss      
------------------------------------------------------------
1      BiGRU_Balanced_Regularization  34.95%     3.6217
2      Lightweight_BiLSTM_Balanced_Regularization  27.67%     3.8766

✅ Summary saved to: ../testing/model_comparison_results/summary_20251127_034440.txt
✅ All results saved to: ../testing/model_comparison_results/

🎉 Testing Complete!
